# 🧬 oligoN-design — Interactive Notebook

**oligoN-design** is a Python-based bioinformatics tool that automates the
design of specific oligonucleotides (FISH probes or PCR primers) from large,
heterogeneous sequence datasets. It is optimised for the Small Sub-Unit rDNA
(18S and 16S), but can be applied to any gene.

> **Reference:** Sandin MM *et al.* (2025) *Mol. Ecol. Resour.* 26(3): e70140.  
> doi:[10.1111/1755-0998.70140](https://doi.org/10.1111/1755-0998.70140) ·
> [GitHub](https://github.com/MiguelMSandin/oligoN-design)

---

## Running on Binder

This notebook is designed to run on **[Binder](https://mybinder.org)** — no
installation required. Binder has already provisioned a server with
oligoN-design, agrep, mafft, and hmmer pre-installed and ready to use.

> ⚠️ **Binder sessions are temporary.** The server and all files on it are
> deleted when your session closes or times out (~10 minutes of inactivity).
> **Download your results using the button in Cell 10 before closing the tab.**

### Uploading your FASTA files

Because Binder runs on a remote server, your local files are not available
there automatically. **Cell 3** provides upload widgets — click the
*Upload* button to send your `.fasta` files from your computer to the server.

---

## How to use this notebook

Work through the cells **from top to bottom**. Each code cell is preceded by
a markdown cell explaining what it does and why.

| Cell | Purpose |
|------|---------|
| 1 | Verify that oligoN-design and all dependencies are present |
| 2 | Choose a workflow mode (Unsupervised or Supervised) |
| 3 | **Upload** your FASTA files and set the output location |
| 4 | (Optional) Build target/excluding files with `sequenceSelect` |
| 5 | Set pipeline parameters |
| 6 | Preview the exact shell commands that will be run |
| 7 | Run the pipeline and stream live output |
| 8 | Browse and inspect the output tables |
| 9 | Plot a summary of candidate oligonucleotides |
| 10 | Display the final selected oligonucleotides and **download results** |

---

## Background: the two workflows

```
UNSUPERVISED  ──  oligoNdesign -t target.fasta -e excluding.fasta -o prefix
                  • One command, sensible defaults
                  • Selects the N best oligos automatically
                  • Good for a first pass or quick exploration

SUPERVISED    ──  findOligo  →  testOligo  →  (testThorough)
                            →  rateAccess  →  (logStats / filterLog)
                            →  selectOligo
                  • Full control over every threshold and parameter
                  • Lets you inspect intermediate results
                  • Recommended for publication-ready designs
```


---
## Cell 1 — Environment check

When running on Binder, the full environment is pre-configured automatically
from the repository's `environment.yml` and `apt.txt` files — nothing needs
to be installed manually.

This cell simply verifies that every expected tool is present and reachable
on `PATH`. All entries should show ✅. If any show ❌, the Binder image may
not have built correctly; try relaunching from the Binder badge in the README.


In [ ]:
import shutil

# ── Check a single executable and print a friendly status line ──────────────
def check_tool(name: str) -> bool:
    """Return True if `name` is found on PATH, False otherwise."""
    path = shutil.which(name)
    if path:
        print(f"  ✅  {name:20s}  →  {path}")
    else:
        print(f"  ❌  {name:20s}  →  NOT FOUND")
    return path is not None


# ── oligoN-design functions ──────────────────────────────────────────────────
print("oligoN-design functions")
print("=" * 55)
oligon_tools = [
    "oligoNdesign",    # unsupervised wrapper (runs the full pipeline)
    "sequenceSelect",  # filter sequences from a FASTA by header pattern
    "findOligo",       # find candidate oligonucleotides
    "testOligo",       # fast mismatch test using agrep
    "testThorough",    # thorough mismatch test (slower, more detail)
    "rateAccess",      # rate SSU secondary-structure accessibility
    "identifyRegions", # group overlapping sliding-window oligos into regions
    "getHomolog",      # retrieve homologous regions from the excluding file
    "getHomologStats", # pairwise similarity histogram of retrieved regions
    "logStats",        # summary statistics of a candidate oligo table
    "filterLog",       # filter a candidate oligo table by column thresholds
    "selectOligo",     # select the best-scoring candidate oligos
]
tool_status = {t: check_tool(t) for t in oligon_tools}

# ── External dependencies called internally by oligoN-design ─────────────────
print()
print("External dependencies")
print("=" * 55)
for dep in ["agrep", "mafft", "hmmbuild", "hmmsearch", "python3"]:
    check_tool(dep)

# ── Python packages ──────────────────────────────────────────────────────────
print()
print("Python packages")
print("=" * 55)
for pkg in ["ipywidgets", "ipyfilechooser", "pandas", "matplotlib"]:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "unknown version")
        print(f"  ✅  {pkg:20s}  →  {ver}")
    except ImportError:
        print(f"  ❌  {pkg:20s}  →  NOT FOUND")

# ── Summary ──────────────────────────────────────────────────────────────────
print()
n_missing = sum(1 for ok in tool_status.values() if not ok)
if n_missing == 0:
    print("✅  All oligoN-design tools found. The environment is ready.")
else:
    print(
        f"❌  {n_missing} tool(s) not found.\n"
        "    This Binder image may not have built correctly.\n"
        "    Try closing this tab and relaunching from the Binder badge in the README."
    )


---
## Cell 2 — Choose a workflow

Select whether to run the **Unsupervised** or the **Supervised** workflow.
Your choice here controls which parameter panels appear in Cell 5 and
which commands are assembled in Cell 6.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ── Shared widget styling ────────────────────────────────────────────────────
# These two variables are reused across all cells so the layout stays consistent
LABEL_WIDTH = {"description_width": "210px"}
INPUT_WIDTH  = widgets.Layout(width="580px")

# ── Workflow selector toggle ─────────────────────────────────────────────────
workflow_selector = widgets.ToggleButtons(
    options=[
        ("🚀  Unsupervised  (oligoNdesign wrapper)", "unsupervised"),
        ("🔬  Supervised  (step-by-step)",           "supervised"),
    ],
    value="unsupervised",
    description="Workflow mode:",
    style=LABEL_WIDTH,
    layout=INPUT_WIDTH,
    button_style="info",
)

# Short description shown below the toggle, updates when the selection changes
workflow_description = widgets.HTML()

def update_workflow_description(change):
    """Update the description box whenever the user switches modes."""
    if change["new"] == "unsupervised":
        workflow_description.value = (
            "<div style='margin-top:8px; padding:10px; background:#e3f2fd;"
            "border-left:4px solid #1976D2; border-radius:4px'>"
            "<b>Unsupervised mode</b> runs the <code>oligoNdesign</code> wrapper "
            "function end-to-end with default settings. It finds specific regions, "
            "tests for mismatches, rates SSU accessibility, and returns the "
            "<i>N</i> best-scoring oligonucleotides — all in a single command. "
            "Ideal for a quick first pass on a new dataset."
            "</div>"
        )
    else:
        workflow_description.value = (
            "<div style='margin-top:8px; padding:10px; background:#e8f5e9;"
            "border-left:4px solid #388E3C; border-radius:4px'>"
            "<b>Supervised mode</b> runs each step of the pipeline individually: "
            "<code>findOligo → testOligo → (testThorough) → rateAccess"
            " → (logStats) → selectOligo</code>. "
            "You have full control over every parameter and can inspect "
            "intermediate files between steps. Recommended for publication-ready designs."
            "</div>"
        )

# Register the callback so the description refreshes on every click
workflow_selector.observe(update_workflow_description, names="value")

# Trigger once to populate the description immediately
update_workflow_description({"new": workflow_selector.value})

display(workflow_selector, workflow_description)


---
## Cell 3 — Upload FASTA files and set the output location

Provide the paths to your two input FASTA files:

| File | Role |
|------|------|
| **Target FASTA** | Sequences *belonging to* the taxonomic group of interest. The designed oligo should match as many of these as possible. |
| **Excluding FASTA** | All *other* sequences (e.g. a comprehensive database). The designed oligo should NOT match these. |

### Two ways to provide a file

Each input has two entry methods — use whichever suits you:

- **Upload button** *(for local files)* — click **Upload file** to send a
  `.fasta` file from your local computer to the Binder server. The file is
  saved into the `uploads/` folder and the path is set automatically.
- **URL box + Fetch button** *(for remote files)* — paste a direct URL to a
  FASTA file (e.g. from Zenodo, Figshare, or a public FTP). Click **⬇ Fetch**
  and the file will be downloaded to the server automatically.
- **Path text box** — if a file is already on the server (e.g. sample data
  shipped with this repository), type or paste its path directly.

> ⚠️ **Large excluding files (> ~500 MB):** browser uploads of very large files
> can be slow or time out. For large reference databases (e.g. PR2 or SILVA)
> prefer the URL option, which downloads the file server-side and is much faster.

After a file is set by any method its sequence count and size are shown
automatically so you can catch obvious mistakes early.


In [ ]:
import pathlib
import urllib.request
import ipywidgets as widgets
from IPython.display import display

# ── Folder where uploaded / fetched files are saved on the server ────────────
# Keeping them in a dedicated sub-folder avoids mixing with pipeline outputs.
UPLOAD_DIR = pathlib.Path("uploads")
UPLOAD_DIR.mkdir(exist_ok=True)


# ── Helper: count sequences in a FASTA file ──────────────────────────────────
def fasta_info(path_str: str) -> str:
    """
    Return a short HTML string describing a FASTA file:
    number of sequences and file size.
    Returns a warning string if the file does not exist or cannot be read.
    """
    p = pathlib.Path(path_str.strip())
    if not path_str.strip():
        return "⬜ &nbsp;<i>(not set)</i>"
    if not p.exists():
        return f"⚠️ &nbsp;<code>{p}</code> — <i>file not found</i>"
    size_kb = p.stat().st_size / 1024
    try:
        n_seqs = sum(1 for line in p.open() if line.startswith(">"))
        return (
            f"✅ &nbsp;<code>{p.name}</code> &nbsp;"
            f"— {n_seqs:,} sequences, {size_kb:,.1f} KB"
        )
    except Exception as exc:
        return (
            f"✅ &nbsp;<code>{p.name}</code> &nbsp;"
            f"— {size_kb:,.1f} KB &nbsp;(could not count sequences: {exc})"
        )


# ── Helper: build one complete file-input row ─────────────────────────────────
def make_file_row(label: str):
    """
    Create a labelled input row with three entry methods:

    1. FileUpload widget  — the user picks a file from their local computer.
       ipywidgets >= 8.x: .value is a tuple of dicts with keys
       'name', 'content' (bytes), 'size', 'type'.
       The bytes are written to UPLOAD_DIR on the server.

    2. URL + Fetch button  — the user pastes a public URL and clicks Fetch.
       The file is downloaded server-side with urllib and saved to UPLOAD_DIR.
       The filename is taken from the last segment of the URL path.

    3. Path text box  — the user types or pastes a path that already exists
       on the server (e.g. sample_data/target.fasta).

    All three methods converge on the same text box value, which is what
    the rest of the notebook reads via the _PathProxy accessor.

    Returns
    -------
    container : widgets.VBox   assembled widget ready for display()
    text_box  : widgets.Text   the underlying path store (read via _PathProxy)
    """

    # ── Method 1: local file upload ───────────────────────────────────────────
    uploader = widgets.FileUpload(
        description="Upload file",
        accept=".fasta,.fa,.fna,.fas",
        multiple=False,
        layout=widgets.Layout(width="200px"),
    )

    # ── Method 2: URL download ────────────────────────────────────────────────
    url_box = widgets.Text(
        value="",
        placeholder="Paste a direct URL to a .fasta file…",
        description="",
        layout=widgets.Layout(width="440px"),
    )

    fetch_button = widgets.Button(
        description="⬇ Fetch",
        button_style="info",
        layout=widgets.Layout(width="90px"),
        tooltip="Download the file at the URL above to the server",
    )

    fetch_status = widgets.HTML("")   # inline feedback for the fetch operation

    # ── Method 3: manual path entry ───────────────────────────────────────────
    # This is also the single source of truth read by the pipeline.
    text_box = widgets.Text(
        value="",
        placeholder="…or type / paste a path already on the server",
        description="",
        layout=widgets.Layout(width="580px"),
    )

    # Shared status line: shows sequence count / file-not-found for text_box
    status = widgets.HTML(fasta_info(""))

    # ── Callback: local upload ────────────────────────────────────────────────
    def on_upload(change):
        """Save uploaded bytes to UPLOAD_DIR and update the path text box."""
        if not uploader.value:
            return
        upload_info = uploader.value[0]
        save_path   = UPLOAD_DIR / upload_info["name"]
        save_path.write_bytes(upload_info["content"])
        _set_path(str(save_path))

    # ── Callback: URL fetch ───────────────────────────────────────────────────
    def on_fetch(_):
        """
        Download the file at url_box.value to UPLOAD_DIR.

        Uses urllib.request.urlretrieve which follows redirects and works
        with http, https, and ftp URLs. The filename is taken from the last
        path segment of the URL (everything after the final '/').
        """
        url = url_box.value.strip()
        if not url:
            fetch_status.value = "⚠️ Please enter a URL first."
            return

        # Derive a local filename from the URL path
        filename  = url.rstrip("/").split("/")[-1] or "downloaded.fasta"
        save_path = UPLOAD_DIR / filename

        fetch_status.value = f"⏳ Downloading <code>{filename}</code> …"
        fetch_button.disabled = True

        try:
            urllib.request.urlretrieve(url, save_path)
            fetch_status.value = (
                f"✅ Saved to <code>{save_path}</code>"
            )
            _set_path(str(save_path))
        except Exception as exc:
            fetch_status.value = f"❌ Download failed: {exc}"
        finally:
            fetch_button.disabled = False

    # ── Callback: manual path edit ────────────────────────────────────────────
    def on_text_change(change):
        """Validate the manually entered path and update the status line."""
        # Clear the upload widget so it cannot silently override the typed path
        uploader.value = ()
        status.value = fasta_info(change["new"])

    # ── Internal helper: set text_box without re-triggering on_text_change ────
    def _set_path(path_str: str):
        """Update the path text box and refresh the status line."""
        text_box.unobserve(on_text_change, names="value")
        text_box.value = path_str
        text_box.observe(on_text_change, names="value")
        status.value = fasta_info(path_str)

    # Register callbacks
    uploader.observe(on_upload, names="value")
    fetch_button.on_click(on_fetch)
    text_box.observe(on_text_change, names="value")

    # ── Assemble the widget layout ─────────────────────────────────────────────
    container = widgets.VBox([
        widgets.HTML(f"<b>{label}</b>"),
        # Row A: local upload
        widgets.HBox([
            uploader,
            widgets.HTML(
                "<span style='line-height:32px; color:#666; font-size:0.85em'>"
                "&nbsp; ① upload from your computer</span>"
            ),
        ]),
        # Row B: URL fetch
        widgets.HTML(
            "<span style='color:#666; font-size:0.85em'>"
            "② paste a public URL and click Fetch</span>"
        ),
        widgets.HBox([url_box, fetch_button]),
        fetch_status,
        # Row C: manual path
        widgets.HTML(
            "<span style='color:#666; font-size:0.85em'>"
            "③ or type a path already on the server</span>"
        ),
        text_box,
        status,
    ])

    return container, text_box


# ── Build the two FASTA input rows ───────────────────────────────────────────
target_row,   target_text   = make_file_row("🎯 Target FASTA")
excl_row,     excl_text     = make_file_row("🚫 Excluding FASTA")


# ── Convenience accessor used by later cells ─────────────────────────────────
# Later cells read  target_path_widget.value  and  excluding_path_widget.value.
# The _PathProxy wraps the underlying text widget so callers don't need to know
# which entry method was used.

class _PathProxy:
    """Thin proxy that exposes a .value property pointing at a Text widget."""
    def __init__(self, text_widget):
        self._w = text_widget

    @property
    def value(self):
        return self._w.value.strip()

target_path_widget    = _PathProxy(target_text)
excluding_path_widget = _PathProxy(excl_text)


# ── Output directory and prefix ───────────────────────────────────────────────
output_dir_text = widgets.Text(
    value="./oligoN_results",
    description="Output directory:",
    placeholder="Created automatically if it does not exist",
    style=LABEL_WIDTH,
    layout=INPUT_WIDTH,
)

output_prefix_widget = widgets.Text(
    value="oligos",
    description="Output prefix:",
    placeholder="Short name prepended to all output files, e.g.  oligos → oligos.tsv",
    style=LABEL_WIDTH,
    layout=INPUT_WIDTH,
)

output_dir_widget = _PathProxy(output_dir_text)


# ── Render everything ─────────────────────────────────────────────────────────
display(
    widgets.HTML("<h4>📂 Input files</h4>"),
    target_row,
    widgets.HTML("<br>"),
    excl_row,
    widgets.HTML("<h4 style='margin-top:16px'>📁 Output location</h4>"),
    output_dir_text,
    widgets.HTML("<br>"),
    output_prefix_widget,
)


---
## Cell 4 — (Optional) Build target / excluding files with `sequenceSelect`

If you already have separate target and excluding FASTA files, **skip this
cell** and go straight to Cell 5.

If you have a single large database (e.g. PR2 or SILVA) and want to split it
into target and excluding files based on a header pattern, enable the checkbox
below. `sequenceSelect` will:

1. Write all sequences whose FASTA header **matches** the pattern → `target.fasta`
2. Write all sequences whose header does **not** match → `excluding.fasta`

The pattern uses standard Python `re` (regular expression) matching against the
full header line (including the `>` character).


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ── Toggle to show/hide the sequenceSelect options ──────────────────────────
run_seqsel_checkbox = widgets.Checkbox(
    value=False,
    description="Split a master database with sequenceSelect (optional)",
    style=LABEL_WIDTH,
    layout=widgets.Layout(width="680px"),
)

# ── Parameters for sequenceSelect ────────────────────────────────────────────
database_path_widget = widgets.Text(
    value="database.fasta",
    description="Master DB FASTA:",
    placeholder="Large reference database to split",
    style=LABEL_WIDTH,
    layout=INPUT_WIDTH,
)

pattern_widget = widgets.Text(
    value="",
    description="Header pattern (regex):",
    placeholder="e.g.  Radiolaria   or   >.*Dinoflagellata",
    style=LABEL_WIDTH,
    layout=INPUT_WIDTH,
)

pattern_hint = widgets.HTML(
    "<small><b>Tip:</b> The pattern is matched against the full header line "
    "(the line starting with <code>&gt;</code>). Matching sequences go to the "
    "<i>target</i> file; all others go to the <i>excluding</i> file. "
    "The output paths come from Cell 3.</small>"
)

# Container that shows/hides the sequenceSelect widgets
seqsel_options_box = widgets.VBox([])

def toggle_seqsel_options(change):
    """Show the sequenceSelect options only when the checkbox is ticked."""
    if change["new"]:
        seqsel_options_box.children = [
            widgets.HTML("<b>sequenceSelect options</b>"),
            database_path_widget,
            pattern_widget,
            pattern_hint,
        ]
    else:
        seqsel_options_box.children = []

run_seqsel_checkbox.observe(toggle_seqsel_options, names="value")

display(run_seqsel_checkbox, seqsel_options_box)


---
## Cell 5 — Pipeline parameters

The parameter panels below adapt to the workflow mode chosen in Cell 2:

- **Unsupervised mode** shows a compact set of the most commonly adjusted options.
- **Supervised mode** shows every individual step with its own parameter block.

Default values match the recommendations in the oligoN-design documentation.
Hover over any slider or text box to read a tooltip explaining the parameter.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ═══════════════════════════════════════════════════════════════════════════
# SECTION A — Unsupervised parameters  (oligoNdesign wrapper)
# ═══════════════════════════════════════════════════════════════════════════

unsup_header = widgets.HTML(
    "<h4 style='color:#1565C0; margin-bottom:4px'>"
    "🚀 Unsupervised — oligoNdesign options</h4>"
    "<p style='color:#555; font-size:0.9em; margin-top:0'>These parameters "
    "are passed directly to the <code>oligoNdesign</code> wrapper. "
    "The wrapper internally calls <code>findOligo</code>, <code>testOligo</code>, "
    "<code>rateAccess</code> and <code>selectOligo</code> with the values below.</p>"
)

# Number of best oligos the wrapper should return
unsup_n_oligos = widgets.BoundedIntText(
    value=4, min=1, max=100,
    description="# oligos to keep:",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Oligo length(s) to search for, given as space-separated integers
unsup_lengths = widgets.Text(
    value="18 20",
    description="Oligo lengths (nt):",
    placeholder="Space-separated, e.g.  16 18 20 22",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Fraction of TARGET sequences that must contain the oligo (≥ threshold → kept)
unsup_target_threshold = widgets.FloatSlider(
    value=0.80, min=0.0, max=1.0, step=0.05,
    description="Target threshold:",
    readout_format=".0%",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Fraction of EXCLUDING sequences that may match (≤ threshold → kept)
unsup_excl_threshold = widgets.FloatSlider(
    value=0.01, min=0.0, max=0.5, step=0.005,
    description="Excluding threshold:",
    readout_format=".1%",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Number of mismatches allowed when testing against the excluding file
unsup_mismatches = widgets.BoundedIntText(
    value=1, min=0, max=5,
    description="Mismatches:",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Whether to call rateAccess to score SSU secondary-structure accessibility
unsup_rate_access = widgets.Checkbox(
    value=True,
    description="Rate SSU secondary-structure accessibility (rateAccess)",
    style=LABEL_WIDTH, layout=widgets.Layout(width="680px"),
)

# Which SSU gene the accessibility table is for (18S eukaryotes vs 16S prokaryotes)
unsup_gene = widgets.Dropdown(
    options=["18S", "16S", "custom"],
    value="18S",
    description="SSU gene:",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Print step-by-step progress messages
unsup_verbose = widgets.Checkbox(
    value=True,
    description="Verbose output (-v)",
    style=LABEL_WIDTH,
)

unsup_box = widgets.VBox([
    unsup_header,
    unsup_n_oligos,
    unsup_lengths,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HTML("<b>Specificity thresholds</b>"),
    unsup_target_threshold,
    unsup_excl_threshold,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HTML("<b>Mismatch testing</b>"),
    unsup_mismatches,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HTML("<b>Accessibility rating</b>"),
    unsup_rate_access,
    unsup_gene,
    widgets.HTML("<hr style='margin:6px 0'>"),
    unsup_verbose,
])


# ═══════════════════════════════════════════════════════════════════════════
# SECTION B — Supervised parameters  (individual steps)
# ═══════════════════════════════════════════════════════════════════════════

sup_header = widgets.HTML(
    "<h4 style='color:#2E7D32; margin-bottom:4px'>"
    "🔬 Supervised — step-by-step options</h4>"
    "<p style='color:#555; font-size:0.9em; margin-top:0'>"
    "Each collapsible block below corresponds to one function in the pipeline. "
    "Steps marked <i>(optional)</i> can be skipped.</p>"
)

# ── findOligo ────────────────────────────────────────────────────────────────
find_header = widgets.HTML(
    "<b>findOligo</b> — search for candidate oligonucleotides<br>"
    "<small style='color:#555'>Scans every possible k-mer of the specified "
    "length(s) across the target file and keeps those that appear in at least "
    "<i>target-threshold</i> fraction of target sequences and in at most "
    "<i>excluding-threshold</i> fraction of excluding sequences.</small>"
)

find_lengths = widgets.Text(
    value="18 20",
    description="Oligo lengths (nt):",
    placeholder="Space-separated list, e.g.  16 18 20 22",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

find_target_threshold = widgets.FloatSlider(
    value=0.80, min=0.0, max=1.0, step=0.05,
    description="Target threshold:",
    readout_format=".0%",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

find_excl_threshold = widgets.FloatSlider(
    value=0.01, min=0.0, max=0.5, step=0.005,
    description="Excluding threshold:",
    readout_format=".1%",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

find_allow_gaps = widgets.Checkbox(
    value=False,
    description="Allow gaps in oligo search (-g)",
    style=LABEL_WIDTH, layout=widgets.Layout(width="500px"),
)

find_allow_ambiguous = widgets.Checkbox(
    value=False,
    description="Allow ambiguous nucleotide codes (-a)",
    style=LABEL_WIDTH, layout=widgets.Layout(width="500px"),
)

find_threads = widgets.BoundedIntText(
    value=1, min=1, max=256,
    description="CPU threads (-c):",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

find_verbose = widgets.Checkbox(
    value=True,
    description="Verbose output (-v)",
    style=LABEL_WIDTH,
)

# ── testOligo ────────────────────────────────────────────────────────────────
test_header = widgets.HTML(
    "<b>testOligo</b> — fast mismatch test using agrep<br>"
    "<small style='color:#555'>For every candidate oligo found by "
    "<code>findOligo</code>, counts how many excluding sequences it matches "
    "when up to <i>N</i> mismatches (including insertions and deletions) "
    "are allowed. Uses the <code>agrep</code> program for speed.</small>"
)

test_mismatches = widgets.BoundedIntText(
    value=1, min=0, max=5,
    description="Mismatches (-m):",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

test_allow_indels = widgets.Checkbox(
    value=True,
    description="Count insertions/deletions as mismatches",
    style=LABEL_WIDTH, layout=widgets.Layout(width="500px"),
)

# ── testThorough (optional) ──────────────────────────────────────────────────
thorough_header = widgets.HTML(
    "<b>testThorough</b> — detailed mismatch analysis <i>(optional)</i><br>"
    "<small style='color:#555'>A slower but more informative mismatch test "
    "that records the position and identity of every mismatch. Run this "
    "only on a pre-filtered set of high-scoring candidates to limit "
    "computation time.</small>"
)

thorough_run = widgets.Checkbox(
    value=False,
    description="Run testThorough after testOligo",
    style=LABEL_WIDTH, layout=widgets.Layout(width="500px"),
)

thorough_mismatches = widgets.BoundedIntText(
    value=1, min=0, max=5,
    description="Mismatches (-m):",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# ── rateAccess ───────────────────────────────────────────────────────────────
rate_header = widgets.HTML(
    "<b>rateAccess</b> — score SSU secondary-structure accessibility<br>"
    "<small style='color:#555'>Uses the empirical accessibility table from "
    "Behrens et al. (2003) to assign each oligo a score reflecting how "
    "accessible its binding region is in the folded SSU rRNA. "
    "Only relevant when targeting the SSU rDNA.</small>"
)

rate_run = widgets.Checkbox(
    value=True,
    description="Run rateAccess",
    style=LABEL_WIDTH, layout=widgets.Layout(width="400px"),
)

rate_gene = widgets.Dropdown(
    options=["18S", "16S", "custom"],
    value="18S",
    description="SSU gene:",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

rate_custom_table = widgets.Text(
    value="",
    description="Custom table path:",
    placeholder="Leave blank to use the built-in 18S or 16S table",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# ── logStats / filterLog (optional) ─────────────────────────────────────────
stats_header = widgets.HTML(
    "<b>logStats / filterLog</b> — inspect and pre-filter candidates "
    "<i>(optional)</i><br>"
    "<small style='color:#555'><code>logStats</code> prints summary statistics "
    "(percentiles, mean) for every numeric column in the candidate table. "
    "<code>filterLog</code> keeps only rows that meet the specified thresholds, "
    "producing a smaller input for <code>selectOligo</code>.</small>"
)

stats_run_logstats = widgets.Checkbox(
    value=True,
    description="Run logStats (print summary statistics)",
    style=LABEL_WIDTH, layout=widgets.Layout(width="600px"),
)

stats_filter_target = widgets.FloatSlider(
    value=0.80, min=0.0, max=1.0, step=0.05,
    description="Min target specificity:",
    readout_format=".0%",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

stats_filter_excl = widgets.FloatSlider(
    value=0.05, min=0.0, max=1.0, step=0.005,
    description="Max excluding matches:",
    readout_format=".1%",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# ── selectOligo ──────────────────────────────────────────────────────────────
select_header = widgets.HTML(
    "<b>selectOligo</b> — pick the best candidates<br>"
    "<small style='color:#555'>Ranks all candidates according to the chosen "
    "criterion and returns the top <i>N</i> oligos as the final output.</small>"
)

select_n = widgets.BoundedIntText(
    value=4, min=1, max=200,
    description="# oligos to select:",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

select_sort_by = widgets.Dropdown(
    options=[
        ("Overall score (recommended)",    "score"),
        ("Target specificity",             "target"),
        ("Excluding matches (ascending)",  "excluding"),
        ("Accessibility score",            "access"),
    ],
    value="score",
    description="Sort / rank by:",
    style=LABEL_WIDTH, layout=INPUT_WIDTH,
)

# Assemble the supervised block
sup_box = widgets.VBox([
    sup_header,
    # findOligo
    widgets.HTML("<hr style='margin:8px 0'>"),
    find_header,
    find_lengths, find_target_threshold, find_excl_threshold,
    find_allow_gaps, find_allow_ambiguous,
    find_threads, find_verbose,
    # testOligo
    widgets.HTML("<hr style='margin:8px 0'>"),
    test_header,
    test_mismatches, test_allow_indels,
    # testThorough (optional)
    widgets.HTML("<hr style='margin:8px 0'>"),
    thorough_header,
    thorough_run, thorough_mismatches,
    # rateAccess
    widgets.HTML("<hr style='margin:8px 0'>"),
    rate_header,
    rate_run, rate_gene, rate_custom_table,
    # logStats / filterLog
    widgets.HTML("<hr style='margin:8px 0'>"),
    stats_header,
    stats_run_logstats, stats_filter_target, stats_filter_excl,
    # selectOligo
    widgets.HTML("<hr style='margin:8px 0'>"),
    select_header,
    select_n, select_sort_by,
])


# ── Dynamically show the correct parameter box ───────────────────────────────
params_output_area = widgets.Output()

def refresh_params(*_):
    """
    Show the Unsupervised or Supervised parameter box depending on
    the current value of workflow_selector (defined in Cell 2).
    """
    params_output_area.clear_output(wait=True)
    with params_output_area:
        if workflow_selector.value == "unsupervised":
            display(unsup_box)
        else:
            display(sup_box)

# Observe the workflow selector from Cell 2 so switching modes updates params
workflow_selector.observe(refresh_params, names="value")
refresh_params()  # populate immediately

display(params_output_area)


---
## Cell 6 — Preview commands

This cell assembles the shell commands that *will* be executed in Cell 7,
based on all the settings chosen above.

**Run this cell to verify the commands before committing to a potentially
long computation.** Nothing is executed yet.


In [ ]:
import shlex
import pathlib

def build_commands() -> list:
    """
    Assemble the ordered list of (label, command_string) tuples
    representing the full pipeline to be executed.

    Uses shlex.quote() to safely handle paths with spaces or special characters.
    """
    # Resolve output directory and prefix from the Cell 3 widgets
    out_dir    = pathlib.Path(output_dir_widget.value.strip())
    prefix     = output_prefix_widget.value.strip()
    out_prefix = str(out_dir / prefix)          # e.g. ./results/oligos

    target_fasta   = target_path_widget.value.strip()
    excl_fasta     = excluding_path_widget.value.strip()

    commands = []  # list of (step_label, shell_command_string) tuples

    # ── Step 0 (optional): sequenceSelect ─────────────────────────────────────
    if run_seqsel_checkbox.value:
        db  = database_path_widget.value.strip()
        pat = pattern_widget.value.strip()

        # Build target file from sequences whose header matches the pattern
        commands.append((
            "sequenceSelect  →  target.fasta",
            (f"sequenceSelect"
             f" -f {shlex.quote(db)}"
             f" -p {shlex.quote(pat)}"
             f" -o {shlex.quote(target_fasta)}"),
        ))

        # Build excluding file from sequences whose header does NOT match (-r flag)
        commands.append((
            "sequenceSelect  →  excluding.fasta",
            (f"sequenceSelect"
             f" -f {shlex.quote(db)}"
             f" -p {shlex.quote(pat)}"
             f" -o {shlex.quote(excl_fasta)}"
             f" -r"),
        ))

    # ── Unsupervised workflow ──────────────────────────────────────────────────
    if workflow_selector.value == "unsupervised":
        cmd = (
            f"oligoNdesign"
            f" -t {shlex.quote(target_fasta)}"
            f" -e {shlex.quote(excl_fasta)}"
            f" -o {shlex.quote(out_prefix)}"
            f" -l {unsup_lengths.value.strip()}"
            f" --target-threshold {unsup_target_threshold.value:.4f}"
            f" --excluding-threshold {unsup_excl_threshold.value:.4f}"
            f" --mismatches {unsup_mismatches.value}"
            f" --n-oligos {unsup_n_oligos.value}"
            f" --gene {unsup_gene.value}"
        )
        if unsup_rate_access.value:
            cmd += " --rate-access"
        if unsup_verbose.value:
            cmd += " -v"

        commands.append(("oligoNdesign  (unsupervised wrapper)", cmd))

    # ── Supervised workflow ────────────────────────────────────────────────────
    else:
        # Step 1: findOligo — discover candidate oligos
        find_cmd = (
            f"findOligo"
            f" -t {shlex.quote(target_fasta)}"
            f" -e {shlex.quote(excl_fasta)}"
            f" -o {shlex.quote(out_prefix)}"
            f" -l {find_lengths.value.strip()}"
            f" --target-threshold {find_target_threshold.value:.4f}"
            f" --excluding-threshold {find_excl_threshold.value:.4f}"
            f" -c {find_threads.value}"
        )
        if find_allow_gaps.value:       find_cmd += " -g"
        if find_allow_ambiguous.value:  find_cmd += " -a"
        if find_verbose.value:          find_cmd += " -v"
        commands.append(("findOligo", find_cmd))

        # Step 2: testOligo — fast mismatch screening with agrep
        # Input is the .tsv produced by findOligo
        find_output_tsv = f"{out_prefix}.tsv"
        test_cmd = (
            f"testOligo"
            f" -f {shlex.quote(find_output_tsv)}"
            f" -e {shlex.quote(excl_fasta)}"
            f" -o {shlex.quote(out_prefix + '_tested')}"
            f" -m {test_mismatches.value}"
        )
        if not test_allow_indels.value:
            test_cmd += " --no-indels"
        commands.append(("testOligo", test_cmd))

        # Step 3 (optional): testThorough — detailed mismatch analysis
        if thorough_run.value:
            thor_cmd = (
                f"testThorough"
                f" -f {shlex.quote(out_prefix + '_tested.tsv')}"
                f" -e {shlex.quote(excl_fasta)}"
                f" -o {shlex.quote(out_prefix + '_thorough')}"
                f" -m {thorough_mismatches.value}"
            )
            commands.append(("testThorough", thor_cmd))

        # Step 4 (optional): rateAccess — SSU accessibility scoring
        if rate_run.value:
            # Choose the most downstream TSV available as input
            if thorough_run.value:
                rate_input_tsv = f"{out_prefix}_thorough.tsv"
            else:
                rate_input_tsv = f"{out_prefix}_tested.tsv"

            rate_cmd = (
                f"rateAccess"
                f" -f {shlex.quote(rate_input_tsv)}"
                f" -o {shlex.quote(out_prefix + '_rated')}"
                f" --gene {rate_gene.value}"
            )
            # If the user provided a custom accessibility table, add it
            if rate_custom_table.value.strip():
                rate_cmd += f" --table {shlex.quote(rate_custom_table.value.strip())}"
            commands.append(("rateAccess", rate_cmd))

        # Step 5 (optional): logStats — print summary statistics
        if stats_run_logstats.value:
            if rate_run.value:
                stats_input_tsv = f"{out_prefix}_rated.tsv"
            elif thorough_run.value:
                stats_input_tsv = f"{out_prefix}_thorough.tsv"
            else:
                stats_input_tsv = f"{out_prefix}_tested.tsv"

            commands.append((
                "logStats  (printed to stdout)",
                f"logStats -f {shlex.quote(stats_input_tsv)}",
            ))

            # filterLog — keep only candidates meeting the thresholds
            filter_cmd = (
                f"filterLog"
                f" -f {shlex.quote(stats_input_tsv)}"
                f" -o {shlex.quote(out_prefix + '_filtered')}"
                f" --min-target {stats_filter_target.value:.4f}"
                f" --max-excluding {stats_filter_excl.value:.4f}"
            )
            commands.append(("filterLog", filter_cmd))

        # Step 6: selectOligo — pick the final best candidates
        # Determine the most processed TSV available
        if stats_run_logstats.value:
            select_input_tsv = f"{out_prefix}_filtered.tsv"
        elif rate_run.value:
            select_input_tsv = f"{out_prefix}_rated.tsv"
        elif thorough_run.value:
            select_input_tsv = f"{out_prefix}_thorough.tsv"
        else:
            select_input_tsv = f"{out_prefix}_tested.tsv"

        select_cmd = (
            f"selectOligo"
            f" -f {shlex.quote(select_input_tsv)}"
            f" -o {shlex.quote(out_prefix + '_selected')}"
            f" -n {select_n.value}"
            f" --sort-by {select_sort_by.value}"
        )
        commands.append(("selectOligo", select_cmd))

    return commands


# ── Print a formatted preview ────────────────────────────────────────────────
commands_to_run = build_commands()

print("Commands that will be executed in Cell 7")
print("=" * 65)
for step_index, (label, cmd) in enumerate(commands_to_run, start=1):
    print(f"\n[Step {step_index}]  {label}")
    # Wrap the command for readability at ~70 chars per line
    print(f"  $ {cmd}")
print("\n" + "=" * 65)
print(f"Output directory : {output_dir_widget.value.strip()}")
print(f"Output prefix    : {output_prefix_widget.value.strip()}")


---
## Cell 7 — Run the pipeline

Click **▶ Run pipeline** to execute the commands assembled in Cell 6.

- The **status label** below the buttons updates immediately on click,
  confirming the button was registered.
- The **log area** streams every line of subprocess output as it is produced.
- Use **⏹ Stop** to terminate the current step immediately.

If a step fails the pipeline stops and prints the exit code. Fix the issue
and click ▶ again.

> ⚠️ **Binder reminder:** go to **Cell 10** and click **⬇ Download results**
> as soon as the pipeline finishes. All server files are permanently deleted
> when the Binder session closes or times out (~10 min of inactivity).


In [ ]:
import subprocess
import threading
import time
import pathlib
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display

# ── Buttons ───────────────────────────────────────────────────────────────────
run_button = widgets.Button(
    description="▶  Run pipeline",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="200px", height="40px"),
)

stop_button = widgets.Button(
    description="⏹  Stop",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(width="120px", height="40px"),
    disabled=True,
)

# ── Persistent progress indicator ────────────────────────────────────────────
# This widget lives *outside* the log area and is always visible, even when
# the log is scrolled or empty. It gives immediate feedback when the button
# is clicked and tracks which step is running.
progress_label = widgets.HTML(
    value="<span style='color:#888'>Ready — click ▶ Run pipeline to start.</span>"
)

# ── Log output area ───────────────────────────────────────────────────────────
# We write to this widget exclusively via pipeline_output.append_stdout(),
# NOT via print() inside a `with pipeline_output:` block. The append_stdout()
# method is a direct widget update that works correctly from any thread and
# any async context, whereas the `with` context manager relies on IPython's
# stdout-capture hook, which is unreliable inside threads and coroutines.
pipeline_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        min_height="100px",
        max_height="500px",
        overflow_y="auto",
        padding="8px",
    )
)

# Module-level handle so stop_button can terminate the running subprocess
_active_process = None


# ── Helper: write a line to the log widget from any thread ───────────────────
def log(text: str):
    """
    Append text to the log output widget.

    Uses append_stdout() rather than print() because append_stdout() is a
    direct widget state update. It is safe to call from background threads
    and does not depend on IPython's stdout-capture hook.
    """
    pipeline_output.append_stdout(text)


# ── Pipeline thread function ──────────────────────────────────────────────────
def _pipeline_thread():
    """
    Runs in a background thread so the Jupyter kernel remains responsive
    (widgets stay interactive, the browser does not freeze).

    Streams subprocess output line by line via log() / append_stdout().
    Stops at the first step that returns a non-zero exit code.
    """
    global _active_process

    out_dir  = pathlib.Path(output_dir_widget.value.strip())
    out_dir.mkdir(parents=True, exist_ok=True)

    # Re-read all widget values now so last-minute changes are captured
    commands = build_commands()

    run_ok  = True
    t_start = time.time()

    log(f"Pipeline started at {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}\n")
    log(f"Output directory  : {out_dir.resolve()}\n")
    log(f"Total steps       : {len(commands)}\n")
    log("=" * 65 + "\n")

    for step_i, (label, cmd) in enumerate(commands, start=1):

        # Update the progress label so it is visible even before log output appears
        progress_label.value = (
            f"<b>⏳ Step {step_i} / {len(commands)}: {label}</b>"
        )

        log(f"\n[Step {step_i}/{len(commands)}]  {label}\n")
        log(f"$ {cmd}\n")
        log("-" * 50 + "\n")

        t_step = time.time()

        try:
            # shell=True so the conda PATH and all oligoN-design scripts are found;
            # stderr merged into stdout so everything appears in one stream.
            _active_process = subprocess.Popen(
                cmd,
                shell=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,          # line-buffered: each line arrives immediately
                cwd=str(out_dir),
            )

            # Stream stdout one line at a time as the subprocess produces it
            for line in _active_process.stdout:
                log(line)

            _active_process.wait()
            elapsed = time.time() - t_step

            if _active_process.returncode == 0:
                log(f"\n✅  Step {step_i} finished in {elapsed:.1f} s\n")
            else:
                log(
                    f"\n❌  Step {step_i} failed "
                    f"(exit code {_active_process.returncode}) "
                    f"after {elapsed:.1f} s\n"
                )
                run_ok = False
                break

        except Exception as exc:
            log(f"\n❌  Unexpected error in step {step_i}: {exc}\n")
            run_ok = False
            break

    # ── Final summary ─────────────────────────────────────────────────────────
    total_elapsed = time.time() - t_start
    log("\n" + "=" * 65 + "\n")

    if run_ok:
        log(
            f"🎉  Pipeline completed in {total_elapsed:.1f} s\n"
            f"    Results in: {out_dir.resolve()}\n"
        )
        progress_label.value = (
            f"<b style='color:green'>✅ Pipeline completed in "
            f"{total_elapsed:.1f} s — proceed to Cell 8 to inspect results, "
            f"or Cell 10 to download.</b>"
        )
    else:
        log(f"⚠️  Pipeline stopped after {total_elapsed:.1f} s.\n")
        progress_label.value = (
            "<b style='color:#c62828'>❌ Pipeline failed — "
            "see the log above for the error.</b>"
        )

    # Re-enable Run, disable Stop
    run_button.disabled  = False
    stop_button.disabled = True


# ── Button callbacks ──────────────────────────────────────────────────────────
def on_run_clicked(_):
    """
    Responds immediately to the button click: clears the log, updates the
    progress label, flips button states, then launches the pipeline in a
    background thread so the kernel stays responsive.
    """
    # These three lines run synchronously in the click callback — the user
    # sees the visual change instantly, confirming the click was registered.
    run_button.disabled  = True
    stop_button.disabled = False
    progress_label.value = "<b>⏳ Starting pipeline…</b>"
    pipeline_output.clear_output()

    # Launch the pipeline in a daemon thread (dies automatically if the
    # kernel is restarted, preventing zombie processes)
    threading.Thread(target=_pipeline_thread, daemon=True).start()


def on_stop_clicked(_):
    """Terminate the active subprocess immediately."""
    global _active_process
    if _active_process and _active_process.poll() is None:
        _active_process.terminate()
        log("\n⏹  Stopped by user.\n")
        progress_label.value = (
            "<b style='color:#e65100'>⏹ Pipeline stopped by user.</b>"
        )
    stop_button.disabled = True
    run_button.disabled  = False


run_button.on_click(on_run_clicked)
stop_button.on_click(on_stop_clicked)

display(
    widgets.HBox([run_button, stop_button]),
    progress_label,
    widgets.HTML("<b>Log</b>"),
    pipeline_output,
)


---
## Cell 8 — Browse output tables

After the pipeline finishes, all output `.tsv` files appear in the dropdown
below. Select a file to load it into a pandas DataFrame and display it as an
interactive table.

Each row in these tables is one candidate oligonucleotide. Columns include the
oligo sequence, its genomic position, the fraction of target / excluding
sequences it matches, mismatch counts, accessibility scores, and a composite
ranking score.


In [ ]:
import pathlib
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

out_dir = pathlib.Path(output_dir_widget.value.strip())

# Collect all .tsv files produced by the pipeline, sorted alphabetically
tsv_files = sorted(out_dir.glob("*.tsv"))

if not tsv_files:
    print(
        f"No .tsv files found in  {out_dir}\n"
        "Make sure the pipeline in Cell 7 completed at least one step."
    )
else:
    # Dropdown listing the available output files
    file_selector = widgets.Dropdown(
        options=[(f.name, str(f)) for f in tsv_files],
        description="Output file:",
        style=LABEL_WIDTH,
        layout=INPUT_WIDTH,
    )

    # Area where the table is rendered
    table_display = widgets.Output()

    def load_and_display_table(change):
        """Load the selected TSV into pandas and display it."""
        table_display.clear_output(wait=True)
        with table_display:
            try:
                df = pd.read_csv(change["new"], sep="\t")
                print(f"{df.shape[0]} rows  ×  {df.shape[1]} columns")
                print(f"Columns: {list(df.columns)}")
                print()
                display(df)
            except Exception as exc:
                print(f"Could not read file: {exc}")

    # Load the first file automatically when the cell runs
    file_selector.observe(load_and_display_table, names="value")
    load_and_display_table({"new": str(tsv_files[0])})

    display(
        widgets.HTML("<h4>📄 Select an output file to inspect</h4>"),
        file_selector,
        table_display,
    )


---
## Cell 9 — Summary plots

Two plots help you assess the quality of candidate oligonucleotides at a glance:

| Plot | What it shows |
|------|--------------|
| **Specificity scatter** | Each point is one oligo, plotted as *target specificity* (y-axis) vs *excluding matches* (x-axis). Dashed lines mark the default thresholds (80% target, 1% excluding). Good candidates cluster in the top-left corner. Points are coloured by accessibility score when available. |
| **Length distribution** | Histogram of oligo lengths in the candidate set. |

The plot is also saved as a PNG in the output directory.


In [ ]:
import pathlib
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from IPython.display import display

out_dir     = pathlib.Path(output_dir_widget.value.strip())
prefix_base = output_prefix_widget.value.strip()

# Look for the most informative TSV in order of processing depth
candidate_files = [
    out_dir / f"{prefix_base}_rated.tsv",
    out_dir / f"{prefix_base}_filtered.tsv",
    out_dir / f"{prefix_base}_thorough.tsv",
    out_dir / f"{prefix_base}_tested.tsv",
    out_dir / f"{prefix_base}.tsv",
]
chosen_file = next((p for p in candidate_files if p.exists()), None)

if chosen_file is None:
    print(
        "No candidate oligo TSV found — run the pipeline first (Cell 7)."
    )
else:
    df = pd.read_csv(chosen_file, sep="\t")
    print(f"Loaded : {chosen_file.name}  ({len(df)} candidate oligos)")
    print(f"Columns: {list(df.columns)}\n")

    # ── Detect column names automatically ─────────────────────────────────────
    # Column names may vary slightly between oligoN-design versions, so we
    # identify them by keywords rather than hard-coding exact names.
    col = {}
    for c in df.columns:
        cl = c.lower()
        if "target" in cl and ("spec" in cl or "frac" in cl):
            col.setdefault("target", c)
        elif "target" in cl:
            col.setdefault("target", c)
        if "excl" in cl or "nontarget" in cl or "nonspecif" in cl:
            col.setdefault("excl", c)
        if "access" in cl:
            col.setdefault("access", c)
        if "length" in cl or c.lower() == "len":
            col.setdefault("length", c)
        if "score" in cl:
            col.setdefault("score", c)

    # ── Build the figure ──────────────────────────────────────────────────────
    fig, (ax_scatter, ax_hist) = plt.subplots(
        1, 2, figsize=(13, 5), constrained_layout=True
    )
    fig.suptitle(
        f"oligoN-design — {chosen_file.name}",
        fontsize=13, fontweight="bold"
    )

    # ── Left panel: specificity scatter ───────────────────────────────────────
    x_col = col.get("excl")
    y_col = col.get("target")
    c_col = col.get("access") or col.get("score")  # colour dimension

    if x_col and y_col:
        scatter_kwargs = dict(alpha=0.7, edgecolors="grey", linewidths=0.4, s=60)
        if c_col and c_col in df.columns:
            # Colour each point by its accessibility (or score) value
            sc = ax_scatter.scatter(
                df[x_col], df[y_col],
                c=df[c_col], cmap="RdYlGn",
                **scatter_kwargs,
            )
            fig.colorbar(sc, ax=ax_scatter, label=c_col, shrink=0.8)
        else:
            ax_scatter.scatter(
                df[x_col], df[y_col],
                color="steelblue",
                **scatter_kwargs,
            )

        # Reference lines for the default thresholds
        ax_scatter.axhline(
            0.80, linestyle="--", color="green", alpha=0.6,
            label="80% target threshold",
        )
        ax_scatter.axvline(
            0.01, linestyle="--", color="red", alpha=0.6,
            label="1% excluding threshold",
        )

        ax_scatter.set_xlabel(x_col, fontsize=11)
        ax_scatter.set_ylabel(y_col, fontsize=11)
        ax_scatter.set_title("Specificity scatter", fontsize=12)
        ax_scatter.legend(fontsize=9, loc="lower right")
    else:
        ax_scatter.text(
            0.5, 0.5,
            "Could not identify target/excluding columns.\nSee Cell 8 for column names.",
            ha="center", va="center", transform=ax_scatter.transAxes,
            fontsize=10, color="grey",
        )
        ax_scatter.set_title("Specificity scatter", fontsize=12)

    # ── Right panel: length distribution ──────────────────────────────────────
    len_col = col.get("length")
    if len_col and len_col in df.columns:
        lengths = df[len_col].dropna().astype(int)
        bins = range(lengths.min(), lengths.max() + 2)
        ax_hist.hist(
            lengths, bins=bins,
            color="steelblue", edgecolor="white", align="left",
        )
        ax_hist.set_xlabel("Oligo length (nt)", fontsize=11)
        ax_hist.set_ylabel("Count", fontsize=11)
        ax_hist.set_title("Oligo length distribution", fontsize=12)
        ax_hist.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    else:
        ax_hist.text(
            0.5, 0.5,
            "Could not identify a 'length' column.\nSee Cell 8 for column names.",
            ha="center", va="center", transform=ax_hist.transAxes,
            fontsize=10, color="grey",
        )
        ax_hist.set_title("Oligo length distribution", fontsize=12)

    # Save to the output directory alongside the other results
    plot_path = out_dir / f"{prefix_base}_summary_plot.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to: {plot_path}")


---
## Cell 10 — Final selected oligonucleotides and download

The table below shows only the oligonucleotides that `selectOligo` chose as
the best candidates — these are the sequences to take forward to the wet lab
for empirical validation.

> **Important:** the output of oligoN-design is a **starting point**, not a
> final answer. Every candidate oligo must be empirically validated (e.g. by
> FISH or PCR on positive and negative control samples) before use in a study.

### ⬇ Downloading your results (Binder users)

Because Binder sessions are temporary, **click the download link at the bottom
of this cell** to save a ZIP archive of the entire output directory to your
computer before closing the tab.


In [ ]:
import base64
import io
import pathlib
import zipfile
import pandas as pd
from IPython.display import display, HTML

out_dir      = pathlib.Path(output_dir_widget.value.strip())
prefix_base  = output_prefix_widget.value.strip()
selected_tsv = out_dir / f"{prefix_base}_selected.tsv"

if not selected_tsv.exists():
    print(
        f"Selected oligos file not found:\n  {selected_tsv}\n\n"
        "Make sure the pipeline completed successfully (Cell 7)."
    )
else:
    df_selected = pd.read_csv(selected_tsv, sep="\t")

    print(f"🏆  {len(df_selected)} selected oligonucleotide(s)")
    print(f"    Source file: {selected_tsv}")
    print()

    # ── Print a human-readable per-oligo summary ──────────────────────────────
    # Try to detect the sequence column (name varies between versions)
    seq_col = next(
        (c for c in df_selected.columns
         if "oligo" in c.lower() or "seq" in c.lower() or c.lower() == "probe"),
        None,
    )

    for row_idx, row in df_selected.iterrows():
        print(f"  Oligo {row_idx + 1}")
        for col_name, val in row.items():
            print(f"    {col_name:30s}: {val}")
        print()

    # ── Render as a styled HTML table ────────────────────────────────────────
    display(HTML("<h4>📋 Selected oligos — full table</h4>"))
    display(
        df_selected.style
            .background_gradient(cmap="YlGn", axis=0)
            .set_table_styles([
                {"selector": "th",
                 "props": [("background-color", "#37474F"),
                            ("color", "white"),
                            ("font-weight", "bold"),
                            ("padding", "6px 10px")]},
                {"selector": "td", "props": [("padding", "5px 10px")]},
            ])
    )

    # ── Print FASTA-formatted sequences for quick copy-paste ─────────────────
    if seq_col:
        print("\nFASTA-formatted sequences (copy-paste ready):\n")
        for row_idx, row in df_selected.iterrows():
            print(f">oligo_{row_idx + 1}")
            print(row[seq_col])
    else:
        print("(Sequence column not automatically detected — see full table above.)")


# ── Download button — pack the entire output directory into a ZIP ─────────────
# This works by encoding the ZIP as a base64 data-URI embedded in an HTML
# anchor tag. Clicking the link triggers a normal browser file download
# without requiring any server-side file serving — it works on Binder,
# classic Jupyter, and JupyterLab alike.

def make_download_link(directory: pathlib.Path, zip_name: str) -> str:
    """
    Compress all files in `directory` into an in-memory ZIP archive and
    return an HTML anchor tag that will download it when clicked.

    Parameters
    ----------
    directory : pathlib.Path
        The folder whose contents should be zipped.
    zip_name : str
        The filename the browser will suggest when saving (e.g. 'results.zip').

    Returns
    -------
    str
        An HTML string containing a styled <a> download link, or an error
        message if the directory is empty or does not yet exist.
    """
    if not directory.exists() or not any(directory.iterdir()):
        return (
            "<p style='color:#c62828'>⚠️  Output directory is empty or does not "
            "exist. Run the pipeline first (Cell 7).</p>"
        )

    # Build the ZIP entirely in memory so no extra temp file is written to disk
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(directory.rglob("*")):
            if file_path.is_file():
                # Store with a relative path so the ZIP unpacks cleanly into
                # a single folder named after the output directory
                zf.write(file_path, file_path.relative_to(directory.parent))

    zip_bytes = buffer.getvalue()
    b64       = base64.b64encode(zip_bytes).decode("utf-8")
    size_kb   = len(zip_bytes) / 1024

    return (
        f"<a href=\"data:application/zip;base64,{b64}\" download=\"{zip_name}\">"
        f"<button style=\"font-size:1.1em; padding:10px 24px; background:#1565C0;"
        f"color:white; border:none; border-radius:6px; cursor:pointer;\">"
        f"⬇&nbsp; Download all results&nbsp; ({size_kb:,.0f} KB)"
        f"</button></a>"
    )


zip_filename  = f"{output_prefix_widget.value.strip()}_results.zip"
download_html = make_download_link(out_dir, zip_filename)

display(HTML(
    "<hr style='margin:20px 0'>"
    "<h4>⬇ Download all results</h4>"
    "<p style='color:#555; max-width:620px'>"
    "Click the button below to save a ZIP of the entire output directory "
    "to your computer. "
    "<b>Do this before closing the Binder tab</b> — all server files are "
    "permanently deleted when the session ends.</p>"
    + download_html
))
